In [1]:
from pathlib import Path
import json, csv, zipfile, shutil

In [ ]:
root = Path(r"C:/Users/<USER>/<FOLDER>/dax-pattern-templates")
if root.exists():
    shutil.rmtree(root)
(root/'patterns'/'period-to-date').mkdir(parents=True)
(root/'patterns'/'moving').mkdir(parents=True)
(root/'patterns'/'previous-period').mkdir(parents=True)
(root/'patterns'/'growth').mkdir(parents=True)
(root/'patterns'/'dynamic').mkdir(parents=True)
(root/'docs').mkdir(parents=True)

In [3]:
patterns = {}

In [4]:
def add(folder, code, name, description, dax, kind='value'):
    path = root/'patterns'/folder/f'{code.lower()}.dax'
    content = f'''// Pattern: {code} | {name}
// Purpose: {description}
// Required replacements:
//   [#VALUE_MEASURE#]              Existing measure evaluated by this pattern
//   <DATE_TABLE>[&DATE_FIELD&]     Continuous date column from the marked date table
// Notes:
//   Replace <#MEASURE_NAME#> with the published measure name.
//   Growth patterns return a decimal ratio and should be formatted as a percentage.
//   Validate results at year, quarter, month, and total levels before release.

{dax.strip()}
'''
    path.write_text(content, encoding='utf-8')
    patterns[code] = {'code':code,'name':name,'description':description,'kind':kind,'path':str(path.relative_to(root)).replace('\\','/')}

In [5]:
add('period-to-date','YTD','Year-to-date','Evaluates the base measure from the start of the year through the last visible date.', '''<#MEASURE_NAME#> YTD =
CALCULATE(
  [#VALUE_MEASURE#],
  DATESYTD(
    <DATE_TABLE>[&DATE_FIELD&]
  )
)''')
add('period-to-date','QTD','Quarter-to-date','Evaluates the base measure from the start of the quarter through the last visible date.', '''<#MEASURE_NAME#> QTD =
CALCULATE(
  [#VALUE_MEASURE#],
  DATESQTD(
    <DATE_TABLE>[&DATE_FIELD&]
  )
)''')
add('period-to-date','MTD','Month-to-date','Evaluates the base measure from the start of the month through the last visible date.', '''<#MEASURE_NAME#> MTD =
CALCULATE(
  [#VALUE_MEASURE#],
  DATESMTD(
    <DATE_TABLE>[&DATE_FIELD&]
  )
)''')
add('moving','MAT','Moving annual total','Evaluates the trailing 12-month window ending on the last visible date.', '''<#MEASURE_NAME#> MAT =
VAR _EndDate =
  MAX(
    <DATE_TABLE>[&DATE_FIELD&]
  )
VAR _Result =
  CALCULATE(
    [#VALUE_MEASURE#],
    DATESINPERIOD(
      <DATE_TABLE>[&DATE_FIELD&],
      _EndDate,
      -12,
      MONTH
    )
  )
RETURN
  _Result''')
for code,name,unit in [('PY','Previous year','YEAR'),('PQ','Previous quarter','QUARTER'),('PM','Previous month','MONTH')]:
    add('previous-period',code,name,f'Shifts the current date context backward by one {unit.lower()}.', f'''<#MEASURE_NAME#> {code} =
CALCULATE(
  [#VALUE_MEASURE#],
  DATEADD(
    <DATE_TABLE>[&DATE_FIELD&],
    -1,
    {unit}
  )
)''')
add('previous-period','PYC','Previous year complete','Evaluates the complete calendar year immediately before the year containing the last visible date.', '''<#MEASURE_NAME#> PYC =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _StartDate = DATE(YEAR(_AnchorDate) - 1, 1, 1)
VAR _EndDate = DATE(YEAR(_AnchorDate) - 1, 12, 31)
VAR _Result =
  CALCULATE(
    [#VALUE_MEASURE#],
    REMOVEFILTERS(<DATE_TABLE>),
    DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _StartDate, _EndDate)
  )
RETURN
  _Result''')
add('previous-period','PQC','Previous quarter complete','Evaluates the complete quarter immediately before the quarter containing the last visible date.', '''<#MEASURE_NAME#> PQC =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _CurrentStart = DATE(YEAR(_AnchorDate), INT((MONTH(_AnchorDate) - 1) / 3) * 3 + 1, 1)
VAR _StartDate = EDATE(_CurrentStart, -3)
VAR _EndDate = _CurrentStart - 1
VAR _Result =
  CALCULATE(
    [#VALUE_MEASURE#],
    REMOVEFILTERS(<DATE_TABLE>),
    DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _StartDate, _EndDate)
  )
RETURN
  _Result''')
add('previous-period','PMC','Previous month complete','Evaluates the complete month immediately before the month containing the last visible date.', '''<#MEASURE_NAME#> PMC =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _EndDate = EOMONTH(_AnchorDate, -1)
VAR _StartDate = DATE(YEAR(_EndDate), MONTH(_EndDate), 1)
VAR _Result =
  CALCULATE(
    [#VALUE_MEASURE#],
    REMOVEFILTERS(<DATE_TABLE>),
    DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _StartDate, _EndDate)
  )
RETURN
  _Result''')
add('dynamic','PP','Previous period','Selects previous month, quarter, or year from the active date hierarchy level.', '''<#MEASURE_NAME#> PP =
// Additional replacements:
//   <DATE_TABLE>[&YEAR_FIELD&]
//   <DATE_TABLE>[&QUARTER_FIELD&]
//   <DATE_TABLE>[&MONTH_FIELD&]
VAR _Result =
  SWITCH(
    TRUE(),
    ISINSCOPE(<DATE_TABLE>[&MONTH_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, MONTH)),
    ISINSCOPE(<DATE_TABLE>[&QUARTER_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, QUARTER)),
    ISINSCOPE(<DATE_TABLE>[&YEAR_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, YEAR)),
    BLANK()
  )
RETURN
  _Result''')
add('moving','PYMAT','Previous year moving annual total','Evaluates the comparable trailing 12-month window ending one year before the last visible date.', '''<#MEASURE_NAME#> PYMAT =
VAR _CurrentEnd = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousEnd = EDATE(_CurrentEnd, -12)
VAR _Result =
  CALCULATE(
    [#VALUE_MEASURE#],
    DATESINPERIOD(<DATE_TABLE>[&DATE_FIELD&], _PreviousEnd, -12, MONTH)
  )
RETURN
  _Result''')
for code,name,unit in [('YOY','Year-over-year','YEAR'),('QOQ','Quarter-over-quarter','QUARTER'),('MOM','Month-over-month','MONTH')]:
    add('growth',code,name,f'Calculates percentage change versus the comparable context one {unit.lower()} earlier.', f'''<#MEASURE_NAME#> {code} =
VAR _CurrentValue = [#VALUE_MEASURE#]
VAR _PreviousValue =
  CALCULATE(
    [#VALUE_MEASURE#],
    DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, {unit})
  )
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
add('growth','MATG','Moving annual total growth','Calculates percentage growth between MAT and the comparable MAT ending one year earlier.', '''<#MEASURE_NAME#> MATG =
VAR _CurrentEnd = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousEnd = EDATE(_CurrentEnd, -12)
VAR _CurrentValue = CALCULATE([#VALUE_MEASURE#], DATESINPERIOD(<DATE_TABLE>[&DATE_FIELD&], _CurrentEnd, -12, MONTH))
VAR _PreviousValue = CALCULATE([#VALUE_MEASURE#], DATESINPERIOD(<DATE_TABLE>[&DATE_FIELD&], _PreviousEnd, -12, MONTH))
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
add('dynamic','POP','Period-over-period','Calculates percentage change from the previous month, quarter, or year based on hierarchy scope.', '''<#MEASURE_NAME#> POP =
// Additional replacements:
//   <DATE_TABLE>[&YEAR_FIELD&]
//   <DATE_TABLE>[&QUARTER_FIELD&]
//   <DATE_TABLE>[&MONTH_FIELD&]
VAR _CurrentValue = [#VALUE_MEASURE#]
VAR _PreviousValue =
  SWITCH(
    TRUE(),
    ISINSCOPE(<DATE_TABLE>[&MONTH_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, MONTH)),
    ISINSCOPE(<DATE_TABLE>[&QUARTER_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, QUARTER)),
    ISINSCOPE(<DATE_TABLE>[&YEAR_FIELD&]), CALCULATE([#VALUE_MEASURE#], DATEADD(<DATE_TABLE>[&DATE_FIELD&], -1, YEAR)),
    BLANK()
  )
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
for code,name,base,unit in [('PYTD','Previous year-to-date','DATESYTD','YEAR'),('PQTD','Previous quarter-to-date','DATESQTD','QUARTER'),('PMTD','Previous month-to-date','DATESMTD','MONTH')]:
    add('period-to-date',code,name,f'Evaluates the prior {unit.lower()} through the corresponding elapsed position in the current period.', f'''<#MEASURE_NAME#> {code} =
VAR _CurrentDates = {base}(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousDates = DATEADD(_CurrentDates, -1, {unit})
VAR _Result = CALCULATE([#VALUE_MEASURE#], _PreviousDates)
RETURN
  _Result''')
for code,name,base,unit in [('YOYTD','Year-over-year-to-date','DATESYTD','YEAR'),('QOQTD','Quarter-over-quarter-to-date','DATESQTD','QUARTER'),('MOMTD','Month-over-month-to-date','DATESMTD','MONTH')]:
    add('growth',code,name,f'Calculates percentage change between the current period-to-date value and the comparable prior {unit.lower()}-to-date value.', f'''<#MEASURE_NAME#> {code} =
VAR _CurrentDates = {base}(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousDates = DATEADD(_CurrentDates, -1, {unit})
VAR _CurrentValue = CALCULATE([#VALUE_MEASURE#], _CurrentDates)
VAR _PreviousValue = CALCULATE([#VALUE_MEASURE#], _PreviousDates)
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
add('growth','YTDOPY','Year-to-date over previous year','Calculates percentage change between current YTD and the complete previous calendar year.', '''<#MEASURE_NAME#> YTDOPY =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousStart = DATE(YEAR(_AnchorDate) - 1, 1, 1)
VAR _PreviousEnd = DATE(YEAR(_AnchorDate) - 1, 12, 31)
VAR _CurrentValue = CALCULATE([#VALUE_MEASURE#], DATESYTD(<DATE_TABLE>[&DATE_FIELD&]))
VAR _PreviousValue = CALCULATE([#VALUE_MEASURE#], REMOVEFILTERS(<DATE_TABLE>), DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _PreviousStart, _PreviousEnd))
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
add('growth','QTDOPQ','Quarter-to-date over previous quarter','Calculates percentage change between current QTD and the complete previous quarter.', '''<#MEASURE_NAME#> QTDOPQ =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _CurrentStart = DATE(YEAR(_AnchorDate), INT((MONTH(_AnchorDate) - 1) / 3) * 3 + 1, 1)
VAR _PreviousStart = EDATE(_CurrentStart, -3)
VAR _PreviousEnd = _CurrentStart - 1
VAR _CurrentValue = CALCULATE([#VALUE_MEASURE#], DATESQTD(<DATE_TABLE>[&DATE_FIELD&]))
VAR _PreviousValue = CALCULATE([#VALUE_MEASURE#], REMOVEFILTERS(<DATE_TABLE>), DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _PreviousStart, _PreviousEnd))
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')
add('growth','MTDOPM','Month-to-date over previous month','Calculates percentage change between current MTD and the complete previous month.', '''<#MEASURE_NAME#> MTDOPM =
VAR _AnchorDate = MAX(<DATE_TABLE>[&DATE_FIELD&])
VAR _PreviousEnd = EOMONTH(_AnchorDate, -1)
VAR _PreviousStart = DATE(YEAR(_PreviousEnd), MONTH(_PreviousEnd), 1)
VAR _CurrentValue = CALCULATE([#VALUE_MEASURE#], DATESMTD(<DATE_TABLE>[&DATE_FIELD&]))
VAR _PreviousValue = CALCULATE([#VALUE_MEASURE#], REMOVEFILTERS(<DATE_TABLE>), DATESBETWEEN(<DATE_TABLE>[&DATE_FIELD&], _PreviousStart, _PreviousEnd))
VAR _Result = DIVIDE(_CurrentValue - _PreviousValue, _PreviousValue)
RETURN
  _Result''','percentage')

In [6]:
readme = '''# DAX Pattern Templates

A Git-friendly library of reusable, commented DAX time-intelligence templates. Each pattern is stored in its own `.dax` file so changes are easy to review, test, and merge.

## Included patterns

- Period to date: YTD, QTD, MTD, PYTD, PQTD, PMTD
- Moving periods: MAT, PYMAT
- Previous periods: PY, PQ, PM, PYC, PQC, PMC
- Dynamic periods: PP, POP
- Growth: YOY, QOQ, MOM, MATG, YOYTD, QOQTD, MOMTD, YTDOPY, QTDOPQ, MTDOPM

## Placeholder contract

- `[#VALUE_MEASURE#]`: an existing base measure
- `<#MEASURE_NAME#>`: the friendly root name of the generated measure
- `<DATE_TABLE>`: the marked date-table name
- `[&DATE_FIELD&]`: the continuous date column
- `[&YEAR_FIELD&]`, `[&QUARTER_FIELD&]`, `[&MONTH_FIELD&]`: hierarchy columns used by PP and POP

Example replacement:

```text
<#MEASURE_NAME#>                 -> CPT Code Distribution Value
[#VALUE_MEASURE#]                -> [CPT Code Distribution Value]
<DATE_TABLE>[&DATE_FIELD&]       -> 'dimDates'[DateValue]
<DATE_TABLE>[&YEAR_FIELD&]       -> 'dimDates'[Year]
<DATE_TABLE>[&QUARTER_FIELD&]    -> 'dimDates'[Quarter]
<DATE_TABLE>[&MONTH_FIELD&]      -> 'dimDates'[Month]
```

## Repository workflow

1. Create a branch named `feature/<pattern-or-change>`.
2. Edit one pattern per file whenever practical.
3. Update `manifest.csv` and `CHANGELOG.md` when behavior changes.
4. Validate the DAX against the checklist in `docs/testing.md`.
5. Open a pull request and include the tested model, date range, and expected result.

## Design decisions

- Percentage patterns use `DIVIDE` for safe zero-denominator handling.
- Complete-period patterns remove date-table filters before applying explicit boundaries.
- PP and POP return `BLANK()` when year, quarter, or month is not in scope rather than guessing.
- Manual complete-period boundaries are calendar based. Fiscal calendars require separate fiscal templates.

## Status

This repository is provided as an internal template. Add your organization's approved license or usage notice before publishing externally.
'''
(root/'README.md').write_text(readme, encoding='utf-8')
(root/'CONTRIBUTING.md').write_text('''# Contributing

## Pull request requirements

- Keep placeholders unchanged unless the placeholder contract is intentionally revised.
- Preserve comments that explain purpose, replacements, assumptions, and output type.
- Use underscore-prefixed, singular DAX variable names with proper casing.
- Use `DIVIDE` for ratios.
- Add or update validation notes when calculation behavior changes.
- Do not mix fiscal and calendar behavior in the same template.

## Review checklist

- [ ] DAX parses successfully.
- [ ] Current-period result matches a manually verified result.
- [ ] Prior-period result is aligned to the intended comparison window.
- [ ] Totals and hierarchy levels behave as documented.
- [ ] Blank and zero comparison values do not produce errors.
- [ ] Date filters outside the target period do not leak into complete-period results.
''', encoding='utf-8')
(root/'CHANGELOG.md').write_text('''# Changelog

## 0.1.0 - Initial library

- Added 25 reusable time-intelligence patterns.
- Added placeholder contract, contribution guidance, and testing checklist.
''', encoding='utf-8')
(root/'VERSION').write_text('0.1.0\n', encoding='utf-8')
(root/'.gitignore').write_text('''*.tmp
*.bak
.DS_Store
Thumbs.db
.vscode/
''', encoding='utf-8')
(root/'docs'/'testing.md').write_text('''# Pattern Testing

Test each pattern with a small, independently verifiable base measure.

## Required checks

1. Confirm the date table contains one row per date with no gaps.
2. Confirm the date column is used by the active relationship to the fact table.
3. Test a completed year, quarter, and month.
4. Test a partial year, quarter, and month.
5. Test a leap year and a month-end boundary.
6. Test a zero prior value and a blank prior value.
7. Test row-level results and the visual grand total.
8. For PP and POP, test year, quarter, month, and card contexts.
9. For complete-period patterns, apply a current-period slicer and verify that the prior complete period still resolves.
10. Record the expected and actual result in the pull request.
''', encoding='utf-8')
(root/'docs'/'definitions.md').write_text('''# Definitions

- YTD: Year-to-date
- QTD: Quarter-to-date
- MTD: Month-to-date
- MAT: Moving annual total
- PY: Previous year
- PQ: Previous quarter
- PM: Previous month
- PYC: Previous year complete
- PQC: Previous quarter complete
- PMC: Previous month complete
- PP: Previous period selected from hierarchy scope
- PYMAT: Previous-year moving annual total
- YOY: Year-over-year growth
- QOQ: Quarter-over-quarter growth
- MOM: Month-over-month growth
- MATG: Moving annual total growth
- POP: Period-over-period growth selected from hierarchy scope
- PYTD: Previous year-to-date
- PQTD: Previous quarter-to-date
- PMTD: Previous month-to-date
- YOYTD: Year-over-year-to-date growth
- QOQTD: Quarter-over-quarter-to-date growth
- MOMTD: Month-over-month-to-date growth
- YTDOPY: Year-to-date over complete previous year
- QTDOPQ: Quarter-to-date over complete previous quarter
- MTDOPM: Month-to-date over complete previous month
''', encoding='utf-8')

932

In [7]:
with (root/'manifest.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['code','name','description','kind','path'])
    w.writeheader(); w.writerows(patterns.values())
(root/'manifest.json').write_text(json.dumps(list(patterns.values()), indent=2), encoding='utf-8')

6263

In [ ]:
zip_path = Path(r"C:/Users/<USER>/<FOLDER>/dax-pattern-templates.zip")
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file():
            z.write(p, Path(root.name)/p.relative_to(root))
print(f'Created {len(patterns)} patterns')
print(zip_path)